# Notebook 01 — Get and Clean Data

**Project:** MHD2 — Multi-Hypothesis Distillation for English → Swahili translation  
**Purpose:** Load, inspect, and clean all data needed for the first stage of the MHD pipeline.

### What this notebook does
1. Sets up project paths and checks required files exist.
2. Inspects the FLORES JSONL structure.
3. Extracts FLORES dev / devtest into aligned plain-text files for **evaluation only**.
4. Loads the English monolingual training corpus from `data/raw/english_raw.txt`.
   - **Training data source:** [CC-100 English](https://data.statmt.org/cc-100/) — download `en.txt.xz`, decompress, and place as `data/raw/english_raw.txt`.
   - **FLORES is NOT used as training data.**
5. Cleans the training corpus.
6. Saves `eng_clean.txt`, `eng_clean_1k.txt`, and `eng_clean_10k.txt`.
7. Prints a full summary and sanity previews.

---
**Run this notebook from the `notebooks/` folder.**

## 1. Setup — Paths and Imports

In [1]:
import json
import re
import unicodedata
import random
from pathlib import Path

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# ── Project root (one level up from notebooks/) ──────────────────────────────
PROJECT_ROOT = Path("..").resolve()

FLORES_DIR    = PROJECT_ROOT / "data" / "flores"
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR   = PROJECT_ROOT / "results"

# Create any missing output directories
for d in [FLORES_DIR, RAW_DIR, PROCESSED_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"FLORES_DIR    : {FLORES_DIR}")
print(f"RAW_DIR       : {RAW_DIR}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"RESULTS_DIR   : {RESULTS_DIR}")

PROJECT_ROOT  : C:\Users\nirmi\Desktop\MHD2
FLORES_DIR    : C:\Users\nirmi\Desktop\MHD2\data\flores
RAW_DIR       : C:\Users\nirmi\Desktop\MHD2\data\raw
PROCESSED_DIR : C:\Users\nirmi\Desktop\MHD2\data\processed
RESULTS_DIR   : C:\Users\nirmi\Desktop\MHD2\results


## 2. Check Required Files

In [2]:
files_to_check = {
    "FLORES dev"       : FLORES_DIR / "dev.jsonl",
    "FLORES devtest"   : FLORES_DIR / "devtest.jsonl",
    "English raw corpus": RAW_DIR   / "english_raw.txt",
}

print("File status:")
print("-" * 55)
for label, path in files_to_check.items():
    status = "✔  found" if path.exists() else "✘  MISSING"
    print(f"  {status}  {path.relative_to(PROJECT_ROOT)}")
print("-" * 55)

File status:
-------------------------------------------------------
  ✔  found  data\flores\dev.jsonl
  ✔  found  data\flores\devtest.jsonl
  ✘  MISSING  data\raw\english_raw.txt
-------------------------------------------------------


## 3. Inspect FLORES JSONL Structure

We preview the first 2 rows of `dev.jsonl` to confirm the key names before extraction.

In [3]:
def load_jsonl(path: Path) -> list:
    """Load a JSONL file and return a list of dicts."""
    records = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def extract_flores_pair(record: dict) -> tuple:
    """
    Extract (english, swahili) from a FLORES record.

    Handles multiple known structures:
      - {"source": "...", "reference": ".."}          ← actual MHD FLORES format
      - {"eng_Latn": "...", "swh_Latn": "..."}         ← flat keys
      - {"sentence": {"eng_Latn": ..., "swh_Latn": ...}}
      - {"sentences": {"eng_Latn": ..., "swh_Latn": ...}}
    """
    # Structure 1: source / reference (detected from actual file)
    if "source" in record and "reference" in record:
        return record["source"], record["reference"]

    # Structure 2: flat eng_Latn / swh_Latn
    if "eng_Latn" in record and "swh_Latn" in record:
        return record["eng_Latn"], record["swh_Latn"]

    # Structure 3: nested under "sentence"
    if "sentence" in record:
        sub = record["sentence"]
        if "eng_Latn" in sub and "swh_Latn" in sub:
            return sub["eng_Latn"], sub["swh_Latn"]

    # Structure 4: nested under "sentences"
    if "sentences" in record:
        sub = record["sentences"]
        if "eng_Latn" in sub and "swh_Latn" in sub:
            return sub["eng_Latn"], sub["swh_Latn"]

    raise ValueError(f"Unrecognised FLORES record structure. Keys found: {list(record.keys())}")


# ── Preview first 2 rows ──────────────────────────────────────────────────────
flores_dev_path = FLORES_DIR / "dev.jsonl"
preview_records = []
with open(flores_dev_path, "r", encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        if i >= 2:
            break
        preview_records.append(json.loads(line.strip()))

print("Keys in first record :", list(preview_records[0].keys()))
print()
for i, rec in enumerate(preview_records):
    eng, swh = extract_flores_pair(rec)
    print(f"[Record {i}]")
    print(f"  EN: {eng[:100]}")
    print(f"  SW: {swh[:100]}")
    print()

Keys in first record : ['id', 'flores_id', 'source', 'reference']

[Record 0]
  EN: On Monday, scientists from the Stanford University School of Medicine announced the invention of a n
  SW: Mnamo Jumatatu, wanasayansi kutoka Shule ya Tiba ya Chuo Kikuu cha Stanford walitangaza uvumbuzi wa 

[Record 1]
  EN: Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to pati
  SW: Watafiti wakuu wanasema hili linaweza kuleta ugunduzi wa mapema wa saratani, kifua kikuu, ukimwi na 



## 4. Extract FLORES into Plain-Text Files

From `dev.jsonl`   → `eng_Latn.dev`   and `swh_Latn.dev`  
From `devtest.jsonl` → `eng_Latn.devtest` and `swh_Latn.devtest`  

One sentence per line. **Used for evaluation only — NOT training.**

In [4]:
def extract_flores_split(jsonl_path: Path, split_name: str, output_dir: Path) -> tuple:
    """
    Extract English and Swahili sentences from a FLORES JSONL file.

    Returns (eng_lines, swh_lines) as lists of strings.
    Also writes plain-text files:
        output_dir/eng_Latn.<split_name>
        output_dir/swh_Latn.<split_name>
    """
    records = load_jsonl(jsonl_path)

    eng_lines = []
    swh_lines = []
    for rec in records:
        eng, swh = extract_flores_pair(rec)
        eng_lines.append(eng.strip())
        swh_lines.append(swh.strip())

    eng_out = output_dir / f"eng_Latn.{split_name}"
    swh_out = output_dir / f"swh_Latn.{split_name}"

    eng_out.write_text("\n".join(eng_lines), encoding="utf-8")
    swh_out.write_text("\n".join(swh_lines), encoding="utf-8")

    print(f"  Saved {len(eng_lines):,} lines → {eng_out.relative_to(PROJECT_ROOT)}")
    print(f"  Saved {len(swh_lines):,} lines → {swh_out.relative_to(PROJECT_ROOT)}")

    return eng_lines, swh_lines


print("Extracting FLORES dev split ...")
eng_dev, swh_dev = extract_flores_split(
    FLORES_DIR / "dev.jsonl", "dev", FLORES_DIR
)

print()
print("Extracting FLORES devtest split ...")
eng_devtest, swh_devtest = extract_flores_split(
    FLORES_DIR / "devtest.jsonl", "devtest", FLORES_DIR
)

print()
print("Done.")

Extracting FLORES dev split ...
  Saved 997 lines → data\flores\eng_Latn.dev
  Saved 997 lines → data\flores\swh_Latn.dev

Extracting FLORES devtest split ...
  Saved 1,012 lines → data\flores\eng_Latn.devtest
  Saved 1,012 lines → data\flores\swh_Latn.devtest

Done.


## 5. Validate FLORES Alignment

Each English line must correspond to a Swahili line in both splits.

In [5]:
def count_lines(path: Path) -> int:
    """Count non-empty lines in a plain-text file."""
    with open(path, "r", encoding="utf-8") as fh:
        return sum(1 for line in fh if line.strip())


splits = [("dev", eng_dev, swh_dev), ("devtest", eng_devtest, swh_devtest)]

print("FLORES alignment check:")
print("-" * 45)
for split_name, eng_lines, swh_lines in splits:
    n_eng = len(eng_lines)
    n_swh = len(swh_lines)
    print(f"  {split_name:<10}  EN={n_eng:,}  SW={n_swh:,}", end="  ")
    assert n_eng == n_swh, (
        f"Alignment error in FLORES {split_name}: {n_eng} EN vs {n_swh} SW lines"
    )
    print(f"→  FLORES {split_name} aligned ✔")

print("-" * 45)

FLORES alignment check:
---------------------------------------------
  dev         EN=997  SW=997  →  FLORES dev aligned ✔
  devtest     EN=1,012  SW=1,012  →  FLORES devtest aligned ✔
---------------------------------------------


## 6. Load English Monolingual Raw Corpus

**Source:** [CC-100 English](https://data.statmt.org/cc-100/)  
Download `en.txt.xz`, decompress it, and place the result at `data/raw/english_raw.txt`.

> **FLORES is NOT used here.** It is evaluation-only data.

In [6]:
ENGLISH_RAW_PATH = RAW_DIR / "english_raw.txt"

if not ENGLISH_RAW_PATH.exists():
    print()
    print("=" * 65)
    print("  MISSING: data/raw/english_raw.txt")
    print()
    print("  Add an English monolingual corpus here before continuing.")
    print("  Do NOT use FLORES as training data.")
    print()
    print("  Recommended source: CC-100 English")
    print("  URL: https://data.statmt.org/cc-100/")
    print("  File: en.txt.xz  →  decompress  →  data/raw/english_raw.txt")
    print("=" * 65)
    print()
    # Stop execution of downstream cells gracefully
    raise SystemExit(
        "Notebook halted: english_raw.txt not found. "
        "Add the file and re-run from this cell."
    )

# ── Load raw lines ────────────────────────────────────────────────────────────
print(f"Loading {ENGLISH_RAW_PATH.relative_to(PROJECT_ROOT)} ...")
with open(ENGLISH_RAW_PATH, "r", encoding="utf-8", errors="replace") as fh:
    raw_lines = [line.rstrip("\n") for line in fh]

print(f"  Loaded {len(raw_lines):,} raw lines.")


  MISSING: data/raw/english_raw.txt

  Add an English monolingual corpus here before continuing.
  Do NOT use FLORES as training data.

  Recommended source: CC-100 English
  URL: https://data.statmt.org/cc-100/
  File: en.txt.xz  →  decompress  →  data/raw/english_raw.txt



SystemExit: Notebook halted: english_raw.txt not found. Add the file and re-run from this cell.

c:\Users\nirmi\Desktop\MHD2\venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 7. Clean English Corpus

Cleaning steps applied in order:
1. Strip leading/trailing whitespace.
2. Unicode normalisation (NFKC).
3. Collapse repeated internal spaces.
4. Remove empty lines.
5. Remove lines with **fewer than 4 words** or **more than 80 words**.
6. Remove lines with too many non-ASCII / strange symbols (threshold: >20 % of characters).
7. Deduplicate (preserving first occurrence).
8. Preserve sentence casing.

In [ ]:
# Regex compiled once for efficiency
_MULTI_SPACE   = re.compile(r" {2,}")
_STRANGE_CHARS = re.compile(r"[^\w\s.,;:!?'\"\-()\[\]/&@#%$+<>]")  # non-standard symbols


def clean_line(text: str) -> str | None:
    """
    Clean a single line of English text.

    Returns the cleaned string, or None if the line should be discarded.
    """
    # Step 1 — strip whitespace
    text = text.strip()

    # Step 2 — unicode normalisation
    text = unicodedata.normalize("NFKC", text)

    # Step 3 — collapse repeated spaces
    text = _MULTI_SPACE.sub(" ", text)

    # Step 4 — discard empty
    if not text:
        return None

    # Step 5 — word count filter
    words = text.split()
    if len(words) < 4 or len(words) > 80:
        return None

    # Step 6 — strange symbol filter (>20 % of chars)
    strange_count = len(_STRANGE_CHARS.findall(text))
    if strange_count / len(text) > 0.20:
        return None

    return text  # Step 8 — casing preserved (no lower/upper applied)


def clean_corpus(lines: list) -> tuple:
    """
    Clean a list of raw text lines.

    Returns (cleaned_lines, n_removed) where n_removed = duplicates + invalids.
    """
    cleaned    = []
    seen       = set()
    n_removed  = 0

    for line in lines:
        result = clean_line(line)

        if result is None:
            n_removed += 1
            continue

        # Step 7 — deduplication
        if result in seen:
            n_removed += 1
            continue

        seen.add(result)
        cleaned.append(result)

    return cleaned, n_removed


print("Cleaning corpus ...")
cleaned_lines, n_removed = clean_corpus(raw_lines)
print(f"  Raw lines    : {len(raw_lines):,}")
print(f"  Cleaned lines: {len(cleaned_lines):,}")
print(f"  Removed      : {n_removed:,}")

## 8. Save Cleaned Training Corpus

Three output files:
| File | Content |
|---|---|
| `eng_clean.txt` | All cleaned lines |
| `eng_clean_1k.txt` | Random sample of 1,000 lines (seed 42) |
| `eng_clean_10k.txt` | Random sample of 10,000 lines (seed 42) |

In [ ]:
def save_lines(lines: list, path: Path, label: str) -> None:
    """Write a list of strings to a plain-text file, one per line."""
    path.write_text("\n".join(lines), encoding="utf-8")
    print(f"  Saved {len(lines):,} lines → {path.relative_to(PROJECT_ROOT)}  [{label}]")


def make_sample(lines: list, n: int, label: str, seed: int = RANDOM_SEED) -> list:
    """
    Draw a random sample of n lines (without replacement).
    Warns and returns all lines if fewer than n are available.
    """
    if len(lines) <= n:
        print(
            f"  ⚠ Warning: only {len(lines):,} cleaned lines available "
            f"(requested {n:,} for {label}). Saving all available lines."
        )
        return lines[:]
    rng = random.Random(seed)
    return rng.sample(lines, n)


# ── Full cleaned corpus ───────────────────────────────────────────────────────
CLEAN_PATH     = PROCESSED_DIR / "eng_clean.txt"
CLEAN_1K_PATH  = PROCESSED_DIR / "eng_clean_1k.txt"
CLEAN_10K_PATH = PROCESSED_DIR / "eng_clean_10k.txt"

save_lines(cleaned_lines, CLEAN_PATH, "full")

# ── 1k sample ────────────────────────────────────────────────────────────────
sample_1k = make_sample(cleaned_lines, 1_000, "eng_clean_1k")
save_lines(sample_1k, CLEAN_1K_PATH, "1k sample")

# ── 10k sample ───────────────────────────────────────────────────────────────
sample_10k = make_sample(cleaned_lines, 10_000, "eng_clean_10k")
save_lines(sample_10k, CLEAN_10K_PATH, "10k sample")

print()
print("All cleaned files saved.")

## 9. Summary

In [ ]:
print("=" * 60)
print("  NOTEBOOK 01 — SUMMARY")
print("=" * 60)
print()
print("  English training corpus")
print(f"    Raw lines loaded         : {len(raw_lines):>10,}")
print(f"    Cleaned lines kept       : {len(cleaned_lines):>10,}")
print(f"    Duplicate/invalid removed: {n_removed:>10,}")
print()
print("  FLORES evaluation data (NOT used for training)")
print(f"    dev   examples           : {len(eng_dev):>10,}")
print(f"    devtest examples         : {len(eng_devtest):>10,}")
print()
print("  Output files created")
for path in [
    FLORES_DIR / "eng_Latn.dev",
    FLORES_DIR / "swh_Latn.dev",
    FLORES_DIR / "eng_Latn.devtest",
    FLORES_DIR / "swh_Latn.devtest",
    CLEAN_PATH,
    CLEAN_1K_PATH,
    CLEAN_10K_PATH,
]:
    size_kb = path.stat().st_size / 1024 if path.exists() else 0
    print(f"    {str(path.relative_to(PROJECT_ROOT)):<45}  {size_kb:>8.1f} KB")
print()
print("=" * 60)

## 10. Sanity Previews

In [ ]:
PREVIEW_SEP = "-" * 60

print("First 5 cleaned English training sentences:")
print(PREVIEW_SEP)
for i, line in enumerate(cleaned_lines[:5], 1):
    print(f"  [{i}] {line}")
print()

print("First 3 FLORES English dev sentences:")
print(PREVIEW_SEP)
for i, line in enumerate(eng_dev[:3], 1):
    print(f"  [{i}] {line}")
print()

print("First 3 FLORES Swahili dev reference sentences:")
print(PREVIEW_SEP)
for i, line in enumerate(swh_dev[:3], 1):
    print(f"  [{i}] {line}")
print()